# Analyse MOM6-Solo ISOMIP Output

Use this notebook to inspect a completed MOM6-solo payu archive. It reads model logs, geometry/restart state, and MOM diagnostics, then produces mean-state and evolution plots. The default archive is the validated `ocean3-geometry-noop-smoke` run.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception as exc:
    HAS_PLOTLY = False
    print(f"Plotly unavailable: {exc}")

REPO = Path('/g/data/au88/jr5971/isomip-test-mom6-for-iom3-configs')
sys.path.insert(0, str(REPO / 'tools'))

from isomip_case_builder import archive_summary, latest_archive, latest_numbered_dir


## Select Archive

In [ ]:
ARCHIVE_OR_LAB = Path('/scratch/au88/jr5971/mom6-isomip-gfz/ocean3-geometry-noop-smoke/archive/ocean3-geometry-noop-smoke')

archive = latest_archive(ARCHIVE_OR_LAB)
output = latest_numbered_dir(archive, 'output')
restart = latest_numbered_dir(archive, 'restart')
summary = archive_summary(archive)

print('archive:', archive)
print('output: ', output)
print('restart:', restart)
pd.DataFrame([summary])


In [ ]:
def open_optional(path):
    path = Path(path)
    if path.exists():
        return xr.open_dataset(path, decode_times=False, mask_and_scale=False)
    print(f'Missing: {path}')
    return None

def finite_or_nan(da):
    data = da.where(np.isfinite(da))
    # MOM sometimes writes 9.96921e36 as a missing value without mask metadata being applied.
    return data.where(np.abs(data) < 1.0e30)

def first_existing(ds, names):
    if ds is None:
        return None
    for name in names:
        if name in ds:
            return ds[name]
    return None

ice = open_optional(output / 'ice.nc') if output else None
forcing = open_optional(output / 'forcing.nc') if output else None
prog = open_optional(output / 'ave_prog.nc') if output else None
geom = open_optional(output / 'ocean_geometry.nc') if output else None
shelf_ic = open_optional(output / 'MOM_Shelf_IC.nc') if output else None
shelf_restart = open_optional(restart / 'Shelf.res.nc') if restart else None

for label, ds in [('ice', ice), ('forcing', forcing), ('ave_prog', prog), ('ocean_geometry', geom), ('shelf_restart', shelf_restart)]:
    if ds is not None:
        print(label, dict(ds.sizes))


## Log Scan And Run Summary

In [ ]:
log_hits = summary.get('log_hits', [])
if log_hits:
    print('Potential log issues:')
    for line in log_hits[:30]:
        print(line)
else:
    print('No fatal/error/NaN/reproducing log hits found by the simple scanner.')

print('time_stamp:', summary.get('time_stamp'))
print('exitcode:', summary.get('exitcode'))


## Shelf Geometry State

For very short smoke runs, point diagnostics in `ice.nc` can be all missing-value. In that case, this notebook falls back to `restartNNN/Shelf.res.nc`.

In [ ]:
h_restart = first_existing(shelf_restart, ['h_shelf'])
area_restart = first_existing(shelf_restart, ['shelf_area', 'area_shelf_h'])
mass_restart = first_existing(shelf_restart, ['shelf_mass'])

if h_restart is not None:
    h = finite_or_nan(h_restart.isel(Time=-1) if 'Time' in h_restart.dims else h_restart)
    area = finite_or_nan(area_restart.isel(Time=-1) if 'Time' in area_restart.dims else area_restart)
    mass = finite_or_nan(mass_restart.isel(Time=-1) if 'Time' in mass_restart.dims else mass_restart)
    rows = [{
        'active_cells': int((area > 0).sum()),
        'shelf_area_km2': float(area.sum() / 1.0e6),
        'h_min_m': float(h.where(area > 0).min()),
        'h_max_m': float(h.max()),
        'mass_minus_900h_max_abs': float(abs(mass - 900.0 * h).max()),
    }]
    display(pd.DataFrame(rows))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    h.where(area > 0).plot(ax=axes[0], cmap='viridis')
    axes[0].set_title('restart h_shelf [m]')
    area.where(area > 0).plot(ax=axes[1], cmap='magma')
    axes[1].set_title('restart shelf area [m2]')
    mass.where(area > 0).plot(ax=axes[2], cmap='cividis')
    axes[2].set_title('restart shelf mass [kg m-2]')
    for ax in axes:
        ax.set_aspect('equal')
    plt.show()
else:
    print('No shelf restart state found.')


## Mean State Maps

In [ ]:
def plot_last_map(ds, name, ax, title=None, **kwargs):
    if ds is None or name not in ds:
        ax.set_title(f'missing {name}')
        ax.axis('off')
        return
    da = finite_or_nan(ds[name])
    for dim in list(da.dims):
        if dim.lower() == 'time':
            da = da.isel({dim: -1})
    if 'zl' in da.dims:
        da = da.isel(zl=0)
    if 'Layer' in da.dims:
        da = da.isel(Layer=0)
    da.plot(ax=ax, **kwargs)
    ax.set_title(title or name)
    ax.set_aspect('equal')

fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
plot_last_map(prog, 'temp', axes[0, 0], 'top layer temp')
plot_last_map(prog, 'salt', axes[0, 1], 'top layer salt')
plot_last_map(prog, 'h', axes[0, 2], 'top layer thickness')
plot_last_map(forcing, 'p_surf', axes[1, 0], 'surface pressure')
plot_last_map(ice, 'melt_rate', axes[1, 1], 'melt rate')
plot_last_map(ice, 'ustar_shelf', axes[1, 2], 'ustar shelf')
plt.show()


## Transects

In [ ]:
transect_dim = 'yq'  # use yq/yh or xq/xh depending on the archive grid names
transect_index = None

if prog is not None:
    temp = finite_or_nan(prog['temp']).isel(Time=-1) if 'temp' in prog else None
    salt = finite_or_nan(prog['salt']).isel(Time=-1) if 'salt' in prog else None
    if temp is not None:
        horizontal_dims = [d for d in temp.dims if d not in {'zl', 'zi', 'Time'}]
        ydim = next((d for d in horizontal_dims if d.startswith('y')), horizontal_dims[0])
        idx = transect_index if transect_index is not None else temp.sizes[ydim] // 2
        fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
        temp.isel({ydim: idx}).plot(ax=axes[0], yincrease=False)
        axes[0].set_title(f'temp transect {ydim}={idx}')
        if salt is not None:
            salt.isel({ydim: idx}).plot(ax=axes[1], yincrease=False)
            axes[1].set_title(f'salt transect {ydim}={idx}')
        plt.show()
else:
    print('No ave_prog.nc available.')


## Time Evolution

In [ ]:
series = []
if ice is not None:
    for name in ['melt_rate', 'mass_flux', 'tflux_shelf']:
        if name in ice:
            da = finite_or_nan(ice[name])
            if 'Time' in da.dims:
                series.append((name, da.mean([d for d in da.dims if d != 'Time'], skipna=True)))
if forcing is not None:
    for name in ['p_surf', 'PRCmE']:
        if name in forcing:
            da = finite_or_nan(forcing[name])
            if 'Time' in da.dims:
                series.append((name, da.mean([d for d in da.dims if d != 'Time'], skipna=True)))

if series:
    fig, ax = plt.subplots(figsize=(9, 4))
    for name, da in series:
        da.plot(ax=ax, label=name)
    ax.legend()
    ax.set_title('domain-mean diagnostic time series')
    plt.show()
else:
    print('No time-varying diagnostics with valid values found. For smoke runs, inspect restart state instead.')


## Optional 3-D Restart Geometry

In [ ]:
if HAS_PLOTLY and h_restart is not None and geom is not None:
    h2 = np.asarray(h)
    area2 = np.asarray(area)
    bed = geom['D'].values if 'D' in geom else None
    if bed is not None:
        bed_z = -np.asarray(bed)
        surface_z = np.where(area2 > 0, h2 + 0.0, np.nan)
        base_z = np.where(area2 > 0, 0.0, np.nan)
        fig = go.Figure()
        fig.add_trace(go.Surface(z=bed_z * 20.0, colorscale='Earth', opacity=0.9, name='bed'))
        fig.add_trace(go.Surface(z=base_z * 20.0, colorscale='Blues', opacity=0.7, name='ice/ocean interface'))
        fig.add_trace(go.Surface(z=surface_z * 20.0, colorscale='Ice', opacity=0.5, name='shelf thickness surface'))
        fig.update_layout(title='restart shelf geometry preview', height=700)
        fig.show()
else:
    print('3-D restart view skipped: needs plotly, shelf restart, and ocean_geometry.nc.')
